# Colab runner - Lightweight-EDL-Medical-v2
Run the cells top to bottom in every new session. Code is edited in VS Code, committed with GitHub Desktop and **pulled here**;
never edit code in Colab. Images live on local disk (`/content/data`), results live on Drive.

In [ ]:
# 1. Drive + settings
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = 'Manish-Verma-74'            # change if needed
REPO_NAME   = 'Lightweight-EDL-Medical-v2'
REF         = 'protocol-v1'                # initial frozen tag; for a preregistered cap extension, use the same recorded cap-extension commit/tag on all workers
DRIVE_ROOT  = '/content/drive/MyDrive/Lightweight-EDL-Medical-v2'
WORKER_BACKBONE = 'efficientnet_b0'        # worker 1; use mobilenet_v3_small / shufflenet_v2 on the other workers
WORKER_TAG      = 'efficientnet'           # names this worker's own log file

In [ ]:
# 2. Get the exact code (clone fresh every session) and record the commit
import os, subprocess
%cd /content
!rm -rf {REPO_NAME}
!git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd /content/{REPO_NAME}
!git checkout {REF}
!git rev-parse HEAD
!git status --short

In [ ]:
# 3. Dependencies (does NOT reinstall torch) + environment report
!pip install -q -r requirements.txt
!pip install -q pytest
import torch, torchvision, sklearn, pandas, numpy, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__)
print('sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
# 4. HAM10000 -> LOCAL disk (never read images from Drive)
import os
from getpass import getpass
if 'KAGGLE_API_TOKEN' not in os.environ and not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle API token: ')
!python scripts/download_ham10000.py --dest /content/data/ham10000

In [ ]:
# 5. Unit tests (CPU, a few minutes) - must pass before any experiment
!python -m pytest -q

## First-time setup only (generate the frozen manifests)
Generate in Colab, copy the printed SHA-256 values into `configs/primary.yaml` **in VS Code**, commit and push, then re-run cells 1-3.
The run manifest can be generated locally in VS Code (`python scripts/make_run_manifest.py`).

In [ ]:
# 6a. Split manifest (run ONCE; the frozen study never regenerates it)
!python scripts/make_split.py --data-root /content/data/ham10000
!mkdir -p {DRIVE_ROOT}/protocol_artifacts && cp protocol/split_manifest_v1.csv {DRIVE_ROOT}/protocol_artifacts/

In [ ]:
# 6b. After the hashes are in the config and pulled: verify the data against the frozen manifest
!python scripts/verify_data.py --data-root /content/data/ham10000

In [ ]:
# 6c. OOD manifests (PAD-UFES-20: download the ORIGINAL Mendeley release v1 once, extract to /content/data/pad_ufes_20)
!python scripts/make_ood_manifests.py --pad-root /content/data/pad_ufes_20 --cifar-root /content/data/cifar100
# 6d. Corruption manifest + example grid (inspect protocol/corruption_example_grid.png before freezing the severities)
!python scripts/make_corruption_manifest.py --data-root /content/data/ham10000
!cp protocol/*.csv protocol/*.png {DRIVE_ROOT}/protocol_artifacts/

## Phase 7: smoke tests and the convergence pilot (separate output folders, not part of the 144 runs)

In [ ]:
# 7a. Covering smoke test (a few epochs on a tiny subset); measures epoch and checkpoint-write time
!python scripts/run_experiments.py --phase smoke --drive-root {DRIVE_ROOT} --worker-tag smoke --max-epochs 3 --debug-per-class 10

In [ ]:
# 7b. Five-run convergence pilot (seed 42, train/validation only), then read the decision
!python scripts/run_experiments.py --phase pilot --drive-root {DRIVE_ROOT} --worker-tag pilot --require-gpu T4
!python scripts/pilot_report.py --drive-root {DRIVE_ROOT}
# If an extension is required, resume with the printed cap, e.g.:
# !python scripts/run_experiments.py --phase pilot --drive-root {DRIVE_ROOT} --worker-tag pilot --max-epochs 120

## Phase 10: the 144 primary runs (after the protocol is frozen and tagged)

In [ ]:
# 8. This worker's share of the global manifest; safe to re-run after a disconnect (it resumes)
!python scripts/run_experiments.py --phase primary --drive-root {DRIVE_ROOT} --backbone {WORKER_BACKBONE} --worker-tag {WORKER_TAG} --require-gpu T4

## After training

In [ ]:
# 9. OOD + corruption predictions for completed runs (no retraining)
!python scripts/evaluate_runs.py --drive-root {DRIVE_ROOT} --backbone {WORKER_BACKBONE} --pad-root /content/data/pad_ufes_20

In [ ]:
# 10. (any one worker, after all 144 runs and after all three workers have run evaluation) statistics, plots, efficiency, aggregation
!python scripts/analyze_stats.py --drive-root {DRIVE_ROOT}
!python scripts/plot_results.py --drive-root {DRIVE_ROOT}
!python scripts/run_efficiency.py --drive-root {DRIVE_ROOT} --require-gpu T4
!python scripts/aggregate_results.py --drive-root {DRIVE_ROOT} --require-evaluation